In [1]:
# This is version 3.20 for smFISH analysis
# This code is created by Elias(Qingxu) Guan from Chrsitian Petersen lab. 
# Before using please make sure: 
# Either install everything following readme 
# Follow the following instructions. 
# For windows and linux user
# 1: make sure you create a unique enviornemnt 
# 2: Make sure you have stardist and bigfish/fishquant installed. 
# For apple and apple silicon user: 
# 1. Make sure you create a unique enviornment. 
# 2. If you dont have brew, make sure follow brew instructions and install brew
# 3. Set up c++ complier 
# if you are using apple silicon
# 1. Install stardist following stardist's instruction. 
# 2. Make sure you have installed tensorflow as instructed on tensorflow website. 
# 3. Make sure you have compatible version. https://pypi.org/project/tensorflow-metal/ Here is where you should go. 

In [1]:
# Setting up Your Conditions
# Setting Path
# If you want to Run your code Please change directories here
controlDirectory = None 
experimentDirectory = '/Volumes/Backup Plus/Experiment_results/304_Analysis_results/Experiment'
customFileName = "565.tif"
counterstainDirectory = None
counterstainFileName = "565.tif"

In [3]:
# Setting parameters
kernel_size = (1,1.5,1.5)
# Set the voxel size. This is determined by the pixel size of your microscope. Please contact microscopt manufactuer and convert resolution to voxel size. 
# unit is nm, please change to nm and note this should be the same for control and your experimental image. 
# I specifically allow this code to run different voxel size for control and experimental image, but for a good experiment you should not do it like that. 
control_voxel_size = (361,64,64)
voxel_size = (361,64,64)
minimal_distance = (2,2,2)
# Set the spot size as your expected spot size 
spot_size = (600, 300, 300)
saveSpotInformation = True
counterstain = False 

In [4]:
# Importing packages 
import glob
import os 
import bigfish.detection 
import bigfish.plot
import tifffile
from tqdm import tqdm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
# Here load functions 
# Function1 : Finding all file path

def find_tif_files(directory, pattern="565.tif"):
    matches = []
    for root, dirs, files in os.walk(directory):
        # Skip hidden directories
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for file in files:
            # Skip hidden files
            if file.startswith('.'):
                continue
            if file.endswith(pattern):
                matches.append(os.path.join(root, file))
    return matches
def create_folder_in_same_directory(file_path, folder_name):
    """
    Creates a folder with the specified name in the same directory as the given file.
    If the folder already exists, it returns the existing path.
    """
    # Get the directory of the given file
    directory = os.path.dirname(file_path)
    
    # Define the path for the specified folder
    folder_path = os.path.join(directory, folder_name)
    
    # Check if the folder exists
    if not os.path.exists(folder_path):
        # Create the folder if it doesn't exist
        os.makedirs(folder_path)
        print(f"Created '{folder_name}' folder at: {folder_path}")
    else:
        print(f"'{folder_name}' folder already exists at: {folder_path}")
    
    return folder_path

In [6]:
# If you have control image then read in control images 
if controlDirectory:
    # Create threshold storage 
    controlThresholdCollection = []
    # Finding all tif files in your directory
    controlsmFISHChannelPaths = find_tif_files(controlDirectory,customFileName)
    for i in tqdm(range(len(controlsmFISHChannelPaths))):
        # Get your directory of each single file 
        file_directory = os.path.dirname(controlsmFISHChannelPaths[i])
        # Move to current work directory
        os.chdir(file_directory)
        # Create results folder
        create_folder_in_same_directory(".","results")
        # Aim to save everything in results folder
        os.chdir("results")
        # Read in image
        controlsmFISHChannel = tifffile.imread(controlsmFISHChannelPaths[i])
        # Main Detection function 
        control_spots, control_spots_threshold = bigfish.detection.detect_spots(
                images=controlsmFISHChannel,
                return_threshold=True,
                voxel_size= voxel_size,  # in nanometer (one value per dimension zyx)
                spot_radius= spot_size, # in nanometer (one value per dimension zyx)
                log_kernel_size= kernel_size,
                minimum_distance= minimal_distance)
        # Plot the elbow curve to make sure you have it saved. 
        bigfish.plot.plot_elbow(
                images=controlsmFISHChannel,
                voxel_size=voxel_size,
                spot_radius=spot_size,
                path_output="Elbow.png",
                show=False)
        # Add the threshold to your storage
        controlThresholdCollection.append(control_spots_threshold)
        # Print out the threshold
        print("threshold: {0}".format(control_spots_threshold))

In [7]:
# Create your collection for experiment group thresholds
experimentThresholdCollection = []
#  Find all files for your experiment group
experimentsmFISHChannelPaths = find_tif_files(experimentDirectory, customFileName)
for i in tqdm(range(len(experimentsmFISHChannelPaths))):
    # Same as previous code. 
    file_directory = os.path.dirname(experimentsmFISHChannelPaths[i])
    os.chdir(file_directory)
    create_folder_in_same_directory(".","results")
    os.chdir("results")
    experimentsmFISHChannel = tifffile.imread(experimentsmFISHChannelPaths[i])
    # Function to find your spots and threshold 
    experiment_spots, experiment_spots_threshold = bigfish.detection.detect_spots(
            images=experimentsmFISHChannel,
            return_threshold=True,
            voxel_size= voxel_size,  # in nanometer (one value per dimension zyx)
            spot_radius= spot_size, # in nanometer (one value per dimension zyx)
            log_kernel_size= kernel_size,
            minimum_distance= minimal_distance)
    # Plot your elbow curve
    bigfish.plot.plot_elbow(
            images=experimentsmFISHChannel,
            voxel_size=voxel_size,
            spot_radius=spot_size,
            path_output="Elbow.png",
            show=False)
    # Add your threshold to your collection 
    experimentThresholdCollection.append(experiment_spots_threshold)
    # Save the spots here for future works. 
    print("threshold: {0}".format(experiment_spots_threshold))
    if saveSpotInformation == True:
         with open ("spot_info.txt","w") as file :
             file.write("\r shape: {0}".format(experiment_spots.shape))
             file.write("\r dtype: {0}".format(experiment_spots.dtype))
             file.write("\r threshold: {0}".format(experiment_spots_threshold))
    del experimentsmFISHChannel, experiment_spots

  0%|                                                    | 0/21 [00:00<?, ?it/s]

'results' folder already exists at: results


  5%|█▉                                       | 1/21 [13:08<4:22:52, 788.63s/it]

threshold: 9.818181818181817
'results' folder already exists at: results


 10%|███▉                                     | 2/21 [31:38<5:09:29, 977.36s/it]

...

threshold: 10.181818181818182
'results' folder already exists at: results


 24%|█████████▎                             | 5/21 [1:03:56<3:29:37, 786.10s/it]

threshold: 12.767676767676768
Created 'results' folder at: results


 24%|█████████▎                             | 5/21 [1:03:57<3:24:39, 767.44s/it]


ValueError: 'log_kernel_size' must be a scalar or a sequence with 2 elements.

In [ ]:
import os
import re

def find_threshold_values(root_dir):
    threshold_values = []

    # Walk through all directories and subdirectories
    for dirpath, _, filenames in os.walk(root_dir):
        if "spot_info.txt" in filenames:
            file_path = os.path.join(dirpath, "spot_info.txt")

            # Read the file and extract the threshold value
            with open(file_path, "r") as file:
                for line in file:
                    match = re.search(r"threshold:\s*([\d.]+)", line)
                    if match:
                        threshold_values.append(float(match.group(1)))

    return threshold_values

# Example usage
root_directory = "/Users/eliasguan/Desktop/306_analysis_results/Experiment"  # Change this to your directory path
threshold_numbers = find_threshold_values(root_directory)
print(threshold_numbers)

In [ ]:
thresholds = [controlThresholdCollection, experimentThresholdCollection[3:]]
labels = ['control group Threshold', "Experiment group Threshold"]
plt.figure(figsize=(8, 6))
sns.boxplot(data=thresholds)

# Add title
plt.title("Threshold for Spot Detection")

# Calculate and annotate mean values
for i, dataset in enumerate(thresholds):
    mean_value = np.mean(dataset)
    plt.text(i, mean_value, f'Mean: {mean_value:.2f}', 
             ha='center', va='bottom', color='black')

# Set x-ticks
plt.xticks(ticks=[0, 1], labels=labels)

# Display the plot
plt.show()

In [ ]:
# Create your collection for counterstain group thresholds
if counterstain:
    counterstainThresholdCollection = []
    #  Find all files for your counterstain group
    counterstainsmFISHChannelPaths = find_tif_files(counterstainDirectory, counterstainFileName)
    for i in tqdm(range(len(counterstainsmFISHChannelPaths))):
        # Same as previous code. 
        file_directory = os.path.dirname(counterstainsmFISHChannelPaths[i])
        os.chdir(file_directory)
        create_folder_in_same_directory(".","results")
        os.chdir("results")
        counterstainsmFISHChannel = tifffile.imread(counterstainsmFISHChannelPaths[i])
        # Function to find your spots and threshold 
        counterstain_spots, counterstain_spots_threshold = bigfish.detection.detect_spots(
                images=counterstainsmFISHChannel,
                return_threshold=True,
                voxel_size= voxel_size,  # in nanometer (one value per dimension zyx)
                spot_radius= spot_size, # in nanometer (one value per dimension zyx)
                log_kernel_size= kernel_size,
                minimum_distance= minimal_distance)
        # Plot your elbow curve
        bigfish.plot.plot_elbow(
                images=counterstainsmFISHChannel,
                voxel_size=voxel_size,
                spot_radius=spot_size,
                path_output="Elbow.png",
                show=False)
        # Add your threshold to your collection 
        counterstainThresholdCollection.append(counterstain_spots_threshold)
        # Save the spots here for future works. 
        print("threshold: {0}".format(counterstain_spots_threshold))

In [ ]:
thresholds = [controlThresholdCollection, experimentThresholdCollection, counterstainThresholdCollection]
labels = ['control group Threshold', "Experiment group Threshold", "Counterstain Threshold"]
plt.figure(figsize=(8, 6))
sns.boxplot(data=thresholds)

# Add title
plt.title("Threshold for Spot Detection")

# Calculate and annotate mean values
for i, dataset in enumerate(thresholds):
    mean_value = np.mean(dataset)
    plt.text(i, mean_value, f'Mean: {mean_value:.2f}', 
             ha='center', va='bottom', color='black')

# Set x-ticks
plt.xticks(ticks=[0, 1, 2], labels=labels)

# Display the plot
plt.show()

In [ ]:
# Create your collection for counterstain group thresholds
if counterstain:
    counterstainControlThresholdCollection = []
    #  Find all files for your counterstain group
    counterstainControlsmFISHChannelPaths = find_tif_files(controlDirectory, counterstainFileName)
    for i in tqdm(range(len(counterstainControlsmFISHChannelPaths))):
        # Same as previous code. 
        file_directory = os.path.dirname(counterstainControlsmFISHChannelPaths[i])
        os.chdir(file_directory)
        create_folder_in_same_directory(".","results")
        os.chdir("results")
        counterstainControlsmFISHChannel = tifffile.imread(counterstainControlsmFISHChannelPaths[i])
        # Function to find your spots and threshold 
        counterstainControl_spots, counterstainControl_spots_threshold = bigfish.detection.detect_spots(
                images=counterstainControlsmFISHChannel,
                return_threshold=True,
                voxel_size= voxel_size,  # in nanometer (one value per dimension zyx)
                spot_radius= spot_size, # in nanometer (one value per dimension zyx)
                log_kernel_size= kernel_size,
                minimum_distance= minimal_distance)
        # Plot your elbow curve
        bigfish.plot.plot_elbow(
                images=counterstainControlsmFISHChannel,
                voxel_size=voxel_size,
                spot_radius=spot_size,
                path_output="Elbow.png",
                show=False)
        # Add your threshold to your collection 
        counterstainControlThresholdCollection.append(counterstainControl_spots_threshold)
        # Save the spots here for future works. 
        print("threshold: {0}".format(counterstainControl_spots_threshold))

In [ ]:
thresholds = [controlThresholdCollection, experimentThresholdCollection, counterstainControlThresholdCollection, counterstainThresholdCollection]
labels = ['control group Threshold', "Experiment group Threshold", "Counterstain Control Threshold", "Counterstain Threshold"]
plt.figure(figsize=(12, 6))
sns.boxplot(data=thresholds)

# Add title
plt.title("Threshold for Spot Detection")

# Calculate and annotate mean values
for i, dataset in enumerate(thresholds):
    mean_value = np.mean(dataset)
    plt.text(i, mean_value, f'Mean: {mean_value:.2f}', 
             ha='center', va='bottom', color='black')

# Set x-ticks
plt.xticks(ticks=[0, 1, 2, 3], labels=labels)
plt.savefig("thresholds.png")
# Display the plot
plt.show()

In [14]:
os.getcwd()

'/Users/eliasguan/Desktop/EG_0920_Test_wnt1_incision_amputation/Experiment_dataset/control/0h_incision_Image1/633/results'

In [15]:
plt.savefig("thresholds.png")

<Figure size 640x480 with 0 Axes>